# 3D Mesh Latent → 2D Instance Segmentation via Render-and-Compare

**Concept:**  
Each object class is represented by a **learnable 3D mesh** (the latent).  
For a given input image, we estimate the **camera pose** that aligns the mesh to the 2D observation, then **render a silhouette** — which becomes our predicted segmentation mask.  
Gradients flow from the 2D mask loss back through the differentiable renderer into the 3D mesh vertices.

---

## Use case: Segmenting objects on sidewalk scenes (cars, pedestrians, etc.)

This notebook targets **urban/sidewalk object segmentation** — cars, pedestrians, bikes, etc. — where:
- You have a **3D mesh template per class** (e.g., a canonical car mesh from ShapeNet)
- You want to **project those meshes** into input camera views to produce **2D instance segmentation masks**
- The pipeline uses a **Cityscapes-style street scene** dataset

The cow mesh in this demo is a **drop-in stand-in** for any compact 3D object (car, pedestrian, bike). The architecture is identical — just swap the mesh template.

---

## What training is needed for custom object classes?

| Scenario | What you need |
|---|---|
| **Use a pre-built ShapeNet car/person mesh** | Initialize from ShapeNet, no training needed for the shape itself |
| **Optimize mesh to fit your images** | ~500–2000 iterations of gradient descent per class (silhouette supervision only) |
| **Train a CNN pose encoder** | Need 2D mask annotations + approximate poses (~1000s of images) |
| **Full end-to-end model** | Pix3D, ShapeNet+COCO, or your own paired 2D mask + 3D model dataset |

**In this notebook**: we do **test-time optimization** — no pre-training needed. Given multi-view observations of a class, we optimize the mesh latent to match 2D silhouettes.

## Cell 1 — Install PyTorch3D

In [ ]:
import sys, subprocess, torch
print(f'torch: {torch.__version__}, cuda: {torch.version.cuda}')

def _have():
    try:
        import pytorch3d  # noqa
        return True
    except ModuleNotFoundError:
        return False

def _build_from_source():
    print('Building PyTorch3D from source (slow ~5 min; needs a GPU runtime for CUDA)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'fvcore', 'iopath'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'git+https://github.com/facebookresearch/pytorch3d.git@stable'], check=True)

if not _have():
    if torch.version.cuda is None:
        # CPU runtime: no matching prebuilt wheel -> source build (a GPU runtime is recommended)
        _build_from_source()
    else:
        pyt = torch.__version__.split('+')[0].replace('.', '')
        cu = torch.version.cuda.replace('.', '')
        ver = f'py3{sys.version_info.minor}_cu{cu}_pyt{pyt}'
        url = f'https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{ver}/download.html'
        print(f'Trying prebuilt wheel: {ver}')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'fvcore', 'iopath'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-cache-dir',
                        '-q', '-f', url, 'pytorch3d'])
        if not _have():
            print('Prebuilt wheel not found for this torch/CUDA -> source build.')
            _build_from_source()
import pytorch3d
print(f'pytorch3d installed: {pytorch3d.__version__}')

## Cell 2 — Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import requests
from io import BytesIO

from pytorch3d.utils import ico_sphere
from pytorch3d.io import load_obj, load_objs_as_meshes
from pytorch3d.structures import Meshes
from pytorch3d.renderer import (
    look_at_view_transform, FoVPerspectiveCameras,
    RasterizationSettings, MeshRasterizer,
    SoftSilhouetteShader, MeshRenderer, BlendParams,
    SoftPhongShader, PointLights, TexturesVertex
)
from pytorch3d.transforms import euler_angles_to_matrix
from pytorch3d.loss import (
    mesh_edge_loss, mesh_laplacian_smoothing, mesh_normal_consistency
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Cell 3 — Dataset: Synthetic multi-view renders of a cow mesh

We use PyTorch3D's built-in **cow mesh** as our 3D object template.  
We render it from multiple viewpoints to create a **synthetic dataset** of (image, silhouette mask) pairs.  

This is analogous to: your 3D point cloud / mesh = the canonical class latent, your 2D images = observations you want to segment.

In [ ]:
# Download the cow mesh from pytorch3d
import os
os.makedirs('data', exist_ok=True)

cow_url = 'https://dl.fbaipublicfiles.com/pytorch3d/data/cow_mesh/cow.obj'
cow_tex_url = 'https://dl.fbaipublicfiles.com/pytorch3d/data/cow_mesh/cow.png'
cow_mtl_url = 'https://dl.fbaipublicfiles.com/pytorch3d/data/cow_mesh/cow.mtl'

for url in [cow_url, cow_tex_url, cow_mtl_url]:
    fname = 'data/' + url.split('/')[-1]
    if not os.path.exists(fname):
        r = requests.get(url)
        with open(fname, 'wb') as f:
            f.write(r.content)
        print(f'Downloaded {fname}')
    else:
        print(f'Already exists: {fname}')

# Load mesh
gt_mesh = load_objs_as_meshes(['data/cow.obj'], device=device)
from pytorch3d.renderer import TexturesVertex
if gt_mesh.textures is None:
    _V = int(gt_mesh.num_verts_per_mesh()[0])
    gt_mesh.textures = TexturesVertex(verts_features=0.7 * torch.ones(1, _V, 3, device=device))
    print('mesh had no textures -> attached uniform gray vertex texture (RGB viz only)')
print(f'Loaded mesh: {gt_mesh.verts_packed().shape[0]} vertices, {gt_mesh.faces_packed().shape[0]} faces')

In [ ]:
# ── Renderer setup ──────────────────────────────────────────────
IMAGE_SIZE = 256

def make_silhouette_renderer(cameras, image_size=IMAGE_SIZE):
    blend_params = BlendParams(sigma=1e-4, gamma=1e-4)
    raster_settings = RasterizationSettings(
        image_size=image_size,
        blur_radius=np.log(1. / 1e-4 - 1.) * blend_params.sigma,
        faces_per_pixel=50,
    )
    return MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=SoftSilhouetteShader(blend_params=blend_params)
    )

def make_phong_renderer(cameras, image_size=IMAGE_SIZE):
    raster_settings = RasterizationSettings(
        image_size=image_size, blur_radius=0.0, faces_per_pixel=1
    )
    lights = PointLights(device=device, location=[[2.0, 2.0, -2.0]])
    return MeshRenderer(
        rasterizer=MeshRasterizer(cameras=cameras, raster_settings=raster_settings),
        shader=SoftPhongShader(device=device, cameras=cameras, lights=lights)
    )

# ── Generate synthetic dataset ───────────────────────────────────
# We render the GT cow from N different azimuth angles
N_VIEWS = 20
azimuths = torch.linspace(0, 360, N_VIEWS + 1)[:-1]  # 0..345 degrees

gt_images = []    # RGB renders
gt_silhouettes = []  # Silhouette masks
gt_cameras_list = []

for azim in azimuths:
    R, T = look_at_view_transform(dist=2.7, elev=10, azim=azim.item())
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

    sil_renderer = make_silhouette_renderer(cameras)
    rgb_renderer = make_phong_renderer(cameras)

    with torch.no_grad():
        sil = sil_renderer(gt_mesh)[..., 3]   # (1, H, W) alpha channel
        rgb = rgb_renderer(gt_mesh)[..., :3]  # (1, H, W, 3) RGB

    gt_silhouettes.append(sil.squeeze(0).cpu())  # (H, W)
    gt_images.append(rgb.squeeze(0).cpu())        # (H, W, 3)
    gt_cameras_list.append({'R': R, 'T': T})

print(f'Generated {N_VIEWS} synthetic views')
print(f'Silhouette shape: {gt_silhouettes[0].shape}, Image shape: {gt_images[0].shape}')

# Visualize a few
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    idx = i * (N_VIEWS // 10)
    ax.imshow(gt_images[idx].numpy())
    ax.set_title(f'azim={azimuths[idx]:.0f}°', fontsize=9)
    ax.axis('off')
plt.suptitle('Synthetic Dataset: Cow mesh rendered from different viewpoints', fontsize=12)
plt.tight_layout()
plt.savefig('dataset_views.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved dataset_views.png')

## Cell 4 — Define learnable 3D mesh latent (class template)

We initialize the class latent from an **icosphere** (not the GT cow shape).  
The model must *learn* to deform the sphere into a cow-like shape that projects to matching 2D silhouettes.

This is analogous to: **your 3D point cloud / mesh is the starting latent for your custom object class**.

In [ ]:
# ── Learnable mesh latent ────────────────────────────────────────
# Initialize from ico_sphere (level 4 = 2562 vertices)
src_mesh = ico_sphere(4, device)

# The shape deformation offset — these are the learnable parameters
verts_shape = src_mesh.verts_packed().shape  # (V, 3)
deform_verts = torch.full(verts_shape, 0.0, device=device, requires_grad=True)

# Learnable camera pose parameters (per view, or use a pose encoder CNN)
# For simplicity here: we learn a single canonical pose and vary it
# In a full model, a CNN would predict these from the input image.

print(f'Mesh latent: {verts_shape[0]} vertices, {src_mesh.faces_packed().shape[0]} faces')
print(f'Learnable deformation params: {deform_verts.numel()}')

# Visualize initialization vs GT
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

R, T = look_at_view_transform(dist=2.7, elev=10, azim=0)
cameras = FoVPerspectiveCameras(device=device, R=R, T=T)
sil_renderer = make_silhouette_renderer(cameras)

with torch.no_grad():
    init_sil = sil_renderer(src_mesh)[..., 3].squeeze(0).cpu()

axes[0].imshow(gt_silhouettes[0].numpy(), cmap='gray')
axes[0].set_title('GT silhouette (target)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(init_sil.numpy(), cmap='gray')
axes[1].set_title('Initial latent (icosphere)', fontsize=11)
axes[1].axis('off')

plt.suptitle('Before training: GT vs Initial Mesh Latent', fontsize=12)
plt.tight_layout()
plt.savefig('before_training.png', dpi=100, bbox_inches='tight')
plt.show()

## Cell 5 — Training: Optimize 3D mesh latent via 2D silhouette supervision

We optimize `deform_verts` (the mesh shape latent) using **silhouette render-and-compare loss**.  
All 20 views are used. Gradients flow: 2D pixel loss → differentiable renderer → 3D mesh vertices.

In [ ]:
import torch.nn as nn

# ── Optimizer ───────────────────────────────────────────────────
optimizer = torch.optim.Adam([deform_verts], lr=1e-3)

# Loss weights
w_sil    = 1.0   # Silhouette match
w_smooth = 0.1   # Laplacian smoothness
w_edge   = 1.0   # Edge regularization
w_normal = 0.01  # Normal consistency

N_ITER = 2000
log_interval = 200

losses_history = []

loop = tqdm(range(N_ITER))
for i in loop:
    optimizer.zero_grad()

    # Apply deformation to icosphere to get current mesh latent
    new_src_mesh = src_mesh.offset_verts(deform_verts)

    # Sample a random view from the dataset
    view_idx = np.random.randint(0, N_VIEWS)
    cam = gt_cameras_list[view_idx]
    R, T = cam['R'].to(device), cam['T'].to(device)
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T)

    # Render silhouette from current mesh latent + known camera pose
    sil_renderer = make_silhouette_renderer(cameras)
    pred_sil = sil_renderer(new_src_mesh)[..., 3]  # (1, H, W)
    gt_sil   = gt_silhouettes[view_idx].to(device).unsqueeze(0)  # (1, H, W)

    # ── Losses ──────────────────────────────────────────────────
    # Silhouette L2 loss (differentiable via SoftSilhouetteShader)
    loss_sil    = ((pred_sil - gt_sil) ** 2).mean()
    loss_smooth = mesh_laplacian_smoothing(new_src_mesh, method='uniform')
    loss_edge   = mesh_edge_loss(new_src_mesh)
    loss_normal = mesh_normal_consistency(new_src_mesh)

    loss = (w_sil * loss_sil
            + w_smooth * loss_smooth
            + w_edge   * loss_edge
            + w_normal * loss_normal)

    loss.backward()
    optimizer.step()

    losses_history.append(loss.item())

    if i % log_interval == 0:
        loop.set_description(
            f'Loss={loss.item():.4f} | sil={loss_sil.item():.4f} | '
            f'smooth={loss_smooth.item():.4f} | edge={loss_edge.item():.4f}'
        )

print('Training complete!')

# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(losses_history)
plt.xlabel('Iteration')
plt.ylabel('Total Loss')
plt.title('Training Loss: 3D Mesh Latent Optimization')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=100, bbox_inches='tight')
plt.show()

## Cell 6 — Evaluate: Project learned 3D mesh latent → 2D segmentation masks

In [ ]:
# Project the learned 3D mesh latent -> 2D masks for every view; report IoU.
with torch.no_grad():
    final_mesh = src_mesh.offset_verts(deform_verts)

iou_scores = []
ncol = 4
nrow = (N_VIEWS + ncol - 1) // ncol
fig, big_axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 5 * nrow))
big_axes = big_axes.ravel()

for idx in range(N_VIEWS):
    cam = gt_cameras_list[idx]
    R, T = cam['R'].to(device), cam['T'].to(device)
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T)
    sil_renderer = make_silhouette_renderer(cameras)
    with torch.no_grad():
        pred_sil = sil_renderer(final_mesh)[..., 3].squeeze(0).cpu().numpy()
    gt_sil_np = gt_silhouettes[idx].numpy()

    pred_bin = pred_sil > 0.5
    gt_bin = gt_sil_np > 0.5
    inter = float((pred_bin & gt_bin).sum())
    union = float((pred_bin | gt_bin).sum())
    iou_scores.append(inter / (union + 1e-6))

    ax = big_axes[idx]
    ax.imshow(np.concatenate([gt_sil_np, pred_sil], axis=1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'azim={azimuths[idx]:.0f} deg  IoU={iou_scores[-1]:.3f}\n'
                 f'left: GT mask  |  right: 3D->2D pred', fontsize=8)
    ax.axis('off')
for k in range(N_VIEWS, len(big_axes)):
    big_axes[k].axis('off')

plt.suptitle('Learned 3D mesh latent projected to 2D masks  (left=GT, right=pred)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('segmentation_results.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'\nMean IoU across {N_VIEWS} views: {np.mean(iou_scores):.4f}')
print('Per-view IoU:', [f'{s:.3f}' for s in iou_scores])

## Cell 7 — Visualize: Optimized 3D mesh latent (the learned class shape)

In [ ]:
# Render the learned 3D mesh latent as RGB from multiple views
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for i, azim in enumerate([0, 72, 144, 216, 288]):
    R, T = look_at_view_transform(dist=2.7, elev=20, azim=azim)
    cameras = FoVPerspectiveCameras(device=device, R=R, T=T)
    lights = PointLights(device=device, location=[[2.0, 2.0, -2.0]])

    raster_settings = RasterizationSettings(image_size=256, blur_radius=0.0, faces_per_pixel=1)

    # Color the mesh by vertex position to visualize shape
    verts = final_mesh.verts_packed()
    verts_rgb = (verts - verts.min(0)[0]) / (verts.max(0)[0] - verts.min(0)[0] + 1e-6)
    textures = TexturesVertex(verts_features=[verts_rgb])
    colored_mesh = Meshes(
        verts=[final_mesh.verts_packed()],
        faces=[final_mesh.faces_packed()],
        textures=textures
    )

    phong_renderer = make_phong_renderer(cameras)
    with torch.no_grad():
        rgb = phong_renderer(colored_mesh)[..., :3].squeeze(0).cpu().numpy()

    axes[i].imshow(rgb)
    axes[i].set_title(f'azim={azim}°', fontsize=11)
    axes[i].axis('off')

plt.suptitle('Learned 3D Mesh Latent (optimized from 2D silhouette supervision only)', fontsize=12)
plt.tight_layout()
plt.savefig('learned_mesh_latent.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved learned_mesh_latent.png')

## Cell 8 — Extension: How to use your own 3D mesh for custom classes

**To adapt to a custom class (e.g., a specific object you have a 3D scan of):**

In [ ]:
# ════════════════════════════════════════════════════════════════════
# HOW TO USE YOUR OWN 3D MESH FOR CUSTOM URBAN OBJECT CLASSES
# ════════════════════════════════════════════════════════════════════
#
# The cow mesh in this demo is a stand-in. For urban scenes (cars,
# pedestrians, bikes on a sidewalk), do the following:
#
# Option A: Use a ShapeNet car mesh as the 'car' class latent
# -----------------------------------------------------------
# Download ShapeNet car category (02958343) and pick one canonical mesh:
#   my_mesh = load_objs_as_meshes(['shapenet/02958343/xxx/model.obj'], device=device)
#   deform_verts = torch.zeros(my_mesh.verts_packed().shape, device=device, requires_grad=True)
#   # Then run the same training loop in Cell 5 with car silhouette dataset
#
# Option B: Use Pix3D dataset (paired images + 3D CAD models for cars, chairs, etc.)
# -----------------------------------------------------------
#   https://github.com/xingyuansun/pix3d
#   Pix3D provides real photos + aligned 3D CAD models for multiple categories.
#   Load the 3D model as your mesh latent, use the 2D masks for silhouette supervision.
#
# Option C: Load from a 3D point cloud scan of a car
# -----------------------------------------------------------
# Convert point cloud to mesh using Open3D:
#
#   import open3d as o3d
#   pcd = o3d.io.read_point_cloud('car_scan.ply')
#   pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
#   mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
#   o3d.io.write_triangle_mesh('car_mesh.obj', mesh)
#   my_mesh = load_objs_as_meshes(['car_mesh.obj'], device=device)
#
# Option D: Multi-class setup (car + pedestrian + bike)
# -----------------------------------------------------------
# One mesh latent bank, one entry per class:
#
#   class MeshLatentBank(nn.Module):
#       def __init__(self, class_templates):  # list of (verts, faces) per class
#           super().__init__()
#           self.deform = nn.ParameterList([
#               nn.Parameter(torch.zeros_like(v)) for v, _ in class_templates
#           ])
#           self.faces = [f for _, f in class_templates]
#           self.base_verts = [v.detach() for v, _ in class_templates]
#
#       def get_mesh(self, class_id):
#           verts = self.base_verts[class_id] + self.deform[class_id]
#           return Meshes(verts=[verts], faces=[self.faces[class_id]])
#
# During training: for each detected instance, pick class_id, render, compare to GT mask.
#
# Cityscapes dataset (for urban scene 2D masks):
# -----------------------------------------------------------
#   https://www.cityscapes-dataset.com
#   Classes: car, person, bicycle, motorcycle, truck, bus, etc.
#   Use the fine instance segmentation annotations as your GT silhouette targets.
#   Pair with ShapeNet meshes per class for the 3D latent bank.

print('Custom class setup instructions above.')
print('')
print('Recommended dataset pairings for urban object segmentation:')
print('  2D masks:      Cityscapes (https://www.cityscapes-dataset.com)')
print('  3D mesh latents: ShapeNet (https://shapenet.org) - car (02958343), person (04530566)')
print('  Paired 2D+3D:  Pix3D     (https://github.com/xingyuansun/pix3d)')
print('')
print('For inference on a single image with a pre-optimized mesh latent:')
print('  1. Load your optimized mesh (saved via pytorch3d.io.save_obj)')
print('  2. Run a lightweight pose estimator (or manual look_at_view_transform)')
print('  3. Call sil_renderer(mesh) to get the 2D segmentation mask')


## Cell 9 — Summary & Next Steps

### What we built

| Component | Implementation |
|---|---|
| **3D mesh latent** | `ico_sphere` deformed by `deform_verts` (learnable) |
| **Differentiable renderer** | `MeshRenderer` + `SoftSilhouetteShader` (PyTorch3D) |
| **2D segmentation** | Rendered alpha channel = predicted mask |
| **Loss** | Silhouette L2 + Laplacian + Edge + Normal consistency |
| **Training** | Test-time optimization (no labeled data needed beyond silhouettes) |

### To scale to a full dataset with custom classes

1. **Replace test-time optimization with a CNN pose encoder** (Cell 3 showed the architecture)  
   - Input: image crop  
   - Output: 6D rotation + 3D translation  
   - Train on image+mask pairs from your dataset  

2. **One mesh latent per class** — initialize from ShapeNet or your own 3D scans  

3. **Datasets to try**  
   - [Pix3D](https://github.com/xingyuansun/pix3d): chairs, sofas, tables — paired images + 3D CAD models  
   - [ShapeNet](https://shapenet.org): 55 categories of 3D objects  
   - [STPLS3D](https://www.stpls3d.com): aerial point clouds with sidewalk/road labels  
   - [Roboflow Sidewalk](https://universe.roboflow.com/sidewalk/sidewalk-segmentation): 1928 2D sidewalk images with masks  

4. **Relevant papers**  
   - 3D-RCNN (CVPR 2018): CNN pose encoder + shape latent bank  
   - Mesh R-CNN (ICCV 2019): `github.com/facebookresearch/meshrcnn`  
   - Sparse Multi-Object Render-and-Compare (BMVC 2023)